# ⚠️ Historical timing baseline — AutoCarver **7.0.5**, single process

**Period artifact, not current API.** Exists only to measure the 7.0.5 carving
wall-clock against the 2026 re-run; it needs its own environment
(`uv venv --python 3.11 .venv-705 && uv pip install --python .venv-705 -r
requirements-705.txt` — a full freeze, because installing `autocarver==7.0.5`
today resolves a different scikit-learn and numpy than the measured run).
`MulticlassCarver` here means what `OneVsRestCarver` means on the pinned version.


# Frequency model — 2025 baseline, timed  *(AutoCarver 7.0.5)*

Verbatim copy of `frequency_model.ipynb` cells 0–18 (load → target → weighted
split → `Processor` → column typing → both carving steps), with **nothing changed
but two `time.perf_counter()` wrappers**. Everything after the carving step is
dropped: this notebook exists only to measure the 7.0.5 carving wall-clock against
the 2026 run, on the same machine.

Run it in the isolated old-version environment:

```
uv venv .venv-705 --python 3.11
uv pip install --python .venv-705 "autocarver[jupyter]==7.0.5" xgboost optuna seaborn nbconvert jupyter
.venv-705/Scripts/jupyter nbconvert --to notebook --execute --inplace \
  src/frequency_model_2025_timed.ipynb --ExecutePreprocessor.timeout=-1
```

# Loading Data & Processing

First, let's load the challenge's data and target and join them on ``ID``

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv")
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv")
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

<tmp>/ipykernel_20440\2176795849.py:6: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path+'train_input_Z61KlZo.csv')


x_train (383610, 373)
y_train (383610, 4)


data (383610, 376)


## Defining Target

Has stated in the data dictionary, the number of claims is ``FREQ`` x ``ANNEE_ASSURANCE``, it will be our target variable

In [2]:
target_col = "TARGET"
data[target_col] = (data["FREQ"] * data["ANNEE_ASSURANCE"]).astype(int)
data[target_col].value_counts(normalize=False).sort_index()

TARGET
0    381061
1      2458
2        87
3         2
4         1
5         1
Name: count, dtype: int64

## Stratified Sampling & Weighting

**Stratified Sampling** is applied to ensure same distribution between train (80%) and dev (20%) samples

**Weights** are computed as the inverse frequency of each class of the target, giving more weights to rare classes like (ie observed claims)




**Note:** observations with more than 2 claims are scarce (only 4 of them), they are merged with those that have 2 claims, defining the following multiclass target `Y`:
* `0`: no claim
* `1`: one claim
* `2`: two or more claims

In [3]:
import numpy as np

from collections import Counter
from sklearn.model_selection import train_test_split

y_transform = lambda u: u.where(
    u <= 1, 2
)  # Transform the target variable to ensure binary classification

# Compute class frequencies
class_counts = Counter(y_transform(data[target_col]))
total_samples = len(data[target_col])

# Compute inverse frequency class weights
class_weights = {
    cls: total_samples / (len(class_counts) * count)
    for cls, count in class_counts.items()
}
print("Class weights:", class_weights)

# Assign sample weights based on target values
weights = np.array([class_weights[label] for label in y_transform(data[target_col])])

# Train-test split
x_train, x_dev, y_train, y_dev, w_train, w_dev = train_test_split(
    data,
    data[target_col],
    weights,
    test_size=0.2,
    random_state=42,
    stratify=y_transform(data[target_col]),
)
print("y_train mean", y_train.mean())
print("y_dev mean", y_dev.mean())

Class weights: {0: 0.3355630725789309, 1: 52.0219690805533, 2: 1405.1648351648353}


y_train mean 0.006895023591668622
y_dev mean 0.006921091733792133


In [4]:
y_train.value_counts(normalize=True)

TARGET
0    0.993356
1    0.006406
2    0.000231
5    0.000003
3    0.000003
Name: proportion, dtype: float64

In [5]:
y_dev.value_counts(normalize=True)

TARGET
0    0.993353
1    0.006413
2    0.000209
3    0.000013
4    0.000013
Name: proportion, dtype: float64

# Feature Engineering

The `Processor` class is used to build several new features:
* `ZONE` is converted to `ZONE_REGION`
* Copies of `ALTITUDE_xxx`, `IND_xxx`, `MEN_xxx` and `LOG_xxx` are converted to numerical features: `ALTITUDE_xxx_num`, `IND_xxx_num`, `MEN_xxx_num` and `LOG_xxx_num`
* Vetusty of buildings is computed: `LOG_VETUSTE`
* It learns the distribution of `LOG_TOT`, `LOG_VETUSTE`, `MEN_TOT`, `IND_TOT`, `IND_SNV`,`ALTITUDE_TOT` per `ZONE_REGION`on train sample. It's then used to compute the ratio of each sample to its mean per region (sort of a measure of divergence from the regional mean)
* Per observation, `CA_TOT` and `CA_MEAN` are computed as the sum and mean of `CA1`, `CA2` and `CA3`. `CA_TOT` and `CA_MEAN` are used to compute ratios with  `CA1`, `CA2` and `CA3`
* Open Source data from [Base de Données sur les Incendies de Forêts en France](https://bdiff.agriculture.gouv.fr/) are added: fire extinction rates per `ZONE` in 2023, total surfaces burnt in 2023 and in the 2016-2020 period per `ZONE`, surface burnt over surface of forest per `ZONE` in 2023. Those features are crossed with `NB_CASERNES` and `ZONE_VENT`
* Numerical `KAPITAL_xxx` columns are sumed and maxed out into `KAPITAL_SUM` and `KAPITAL_MAX`
* Temperature columns are crossed with one another
* Binary non-numerical columns are one hot encoded

In [6]:
import warnings
from data_toolkit import Processor

warnings.simplefilter(action="ignore", category=FutureWarning)

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)

# Feature Processing

## Sorting features per data type

* numerical features
* categorical features
* ordinal features (with their respective ordering)

In [7]:
import numpy as np

# get the columns that are categorical
categorical_columns = x_train.select_dtypes(include=["object"]).columns

# get the columns that are numerical
numerical_columns = x_train.select_dtypes(include=["int64", "float64"]).columns

# getting ordinal columns
ordinals = [
    "NB_CASERNES",
    "BDTOPO_BAT_MAX_HAUTEUR",
    "HAUTEUR_MAX",
    "HAUTEUR",
    "BDTOPO_BAT_MAX_HAUTEUR_MAX",
    "MEN_SURF",
    "IND_SNV",
    "IND_INC",
    "IND_Y9",
    "IND_0_Y1",
    "IND",
    "LOG_SOC",
    "LOG_INC",
    "LOG_APA3",
    "LOG_AVA1",
    "MEN_MAIS",
    "MEN_COLL",
    "MEN_FMP",
    "MEN_PROP",
    "MEN_PAUV",
    "MEN",
    "COEFASS",
]
ordinals += [
    "DISTANCE_111",
    "DISTANCE_112",
    "DISTANCE_121",
    "DISTANCE_122",
    "DISTANCE_123",
    "DISTANCE_124",
    "DISTANCE_131",
    "DISTANCE_132",
    "DISTANCE_133",
    "DISTANCE_141",
    "DISTANCE_142",
    "DISTANCE_211",
    "DISTANCE_212",
    "DISTANCE_213",
    "DISTANCE_221",
    "DISTANCE_222",
    "DISTANCE_223",
    "DISTANCE_231",
    "DISTANCE_242",
    "DISTANCE_243",
    "DISTANCE_244",
    "DISTANCE_311",
    "DISTANCE_312",
    "DISTANCE_313",
    "DISTANCE_321",
    "DISTANCE_322",
    "DISTANCE_323",
    "DISTANCE_324",
    "DISTANCE_331",
    "DISTANCE_332",
    "DISTANCE_333",
    "DISTANCE_334",
    "DISTANCE_335",
    "DISTANCE_411",
    "DISTANCE_412",
    "DISTANCE_421",
    "DISTANCE_422",
    "DISTANCE_423",
    "DISTANCE_511",
    "DISTANCE_512",
    "DISTANCE_521",
    "DISTANCE_522",
    "DISTANCE_523",
    "PROPORTION_11",
    "PROPORTION_12",
    "PROPORTION_13",
    "PROPORTION_14",
    "PROPORTION_21",
    "PROPORTION_22",
    "PROPORTION_23",
    "PROPORTION_24",
    "PROPORTION_31",
    "PROPORTION_32",
    "PROPORTION_33",
    "PROPORTION_41",
    "PROPORTION_42",
    "PROPORTION_51",
    "PROPORTION_52",
    "MEN_1IND",
    "MEN_5IND",
    "LOG_A1_A2",
    "LOG_A2_A3",
    "IND_Y1_Y2",
    "IND_Y2_Y3",
    "IND_Y3_Y4",
    "IND_Y4_Y5",
    "IND_Y5_Y6",
    "IND_Y6_Y7",
    "IND_Y7_Y8",
    "IND_Y8_Y9",
    "DISTANCE_1",
    "DISTANCE_2",
    "ALTITUDE_1",
    "ALTITUDE_2",
    "ALTITUDE_3",
    "ALTITUDE_4",
    "ALTITUDE_5",
    "NBJTX25_MM_A",
    "NBJTX25_MMAX_A",
    "NBJTX25_MSOM_A",
    "NBJTX0_MM_A",
    "NBJTX0_MMAX_A",
    "NBJTX0_MSOM_A",
    "NBJTXI27_MM_A",
    "NBJTXI27_MMAX_A",
    "NBJTXI27_MSOM_A",
    "NBJTXS32_MM_A",
    "NBJTXS32_MMAX_A",
    "NBJTXS32_MSOM_A",
    "NBJTXI20_MM_A",
    "NBJTXI20_MMAX_A",
    "NBJTXI20_MSOM_A",
    "NBJTX30_MM_A",
    "NBJTX30_MMAX_A",
    "NBJTX30_MSOM_A",
    "NBJTX35_MM_A",
    "NBJTX35_MMAX_A",
    "NBJTX35_MSOM_A",
    "NBJTN10_MM_A",
    "NBJTN10_MMAX_A",
    "NBJTN10_MSOM_A",
    "NBJTNI10_MM_A",
    "NBJTNI10_MMAX_A",
    "NBJTNI10_MSOM_A",
    "NBJTN5_MM_A",
    "NBJTN5_MMAX_A",
    "NBJTN5_MSOM_A",
    "NBJTNS25_MM_A",
    "NBJTNS25_MMAX_A",
    "NBJTNS25_MSOM_A",
    "NBJTNI15_MM_A",
    "NBJTNI15_MMAX_A",
    "NBJTNI15_MSOM_A",
    "NBJTNI20_MM_A",
    "NBJTNI20_MMAX_A",
    "NBJTNI20_MSOM_A",
    "NBJTNS20_MM_A",
    "NBJTNS20_MMAX_A",
    "NBJTNS20_MSOM_A",
    "NBJTMS24_MM_A",
    "NBJTMS24_MMAX_A",
    "NBJTMS24_MSOM_A",
    "TAMPLIAB_VOR_MM_A",
    "TAMPLIAB_VOR_MMAX_A",
    "TAMPLIM_VOR_MM_A",
    "TAMPLIM_VOR_MMAX_A",
    "TM_VOR_MM_A",
    "TM_VOR_MMAX_A",
    "TMM_VOR_MM_A",
    "TMM_VOR_MMAX_A",
    "TMMAX_VOR_MM_A",
    "TMMAX_VOR_MMAX_A",
    "TMMIN_VOR_MM_A",
    "TMMIN_VOR_MMAX_A",
    "TN_VOR_MM_A",
    "TN_VOR_MMAX_A",
    "TNAB_VOR_MM_A",
    "TNAB_VOR_MMAX_A",
    "TNMAX_VOR_MM_A",
    "TNMAX_VOR_MMAX_A",
    "TX_VOR_MM_A",
    "TX_VOR_MMAX_A",
    "TXAB_VOR_MM_A",
    "TXAB_VOR_MMAX_A",
    "TXMIN_VOR_MM_A",
    "TXMIN_VOR_MMAX_A",
    "NBJFF10_MM_A",
    "NBJFF10_MMAX_A",
    "NBJFF10_MSOM_A",
    "NBJFF16_MM_A",
    "NBJFF16_MMAX_A",
    "NBJFF16_MSOM_A",
    "NBJFF28_MM_A",
    "NBJFF28_MMAX_A",
    "NBJFF28_MSOM_A",
    "NBJFXI3S10_MM_A",
    "NBJFXI3S10_MMAX_A",
    "NBJFXI3S10_MSOM_A",
    "NBJFXI3S16_MM_A",
    "NBJFXI3S16_MMAX_A",
    "NBJFXI3S16_MSOM_A",
    "NBJFXI3S28_MM_A",
    "NBJFXI3S28_MMAX_A",
    "NBJFXI3S28_MSOM_A",
    "NBJFXY8_MM_A",
    "NBJFXY8_MMAX_A",
    "NBJFXY8_MSOM_A",
    "NBJFXY10_MM_A",
    "NBJFXY10_MMAX_A",
    "NBJFXY10_MSOM_A",
    "NBJFXY15_MM_A",
    "NBJFXY15_MMAX_A",
    "NBJFXY15_MSOM_A",
    "FFM_VOR_MM_A",
    "FFM_VOR_MMAX_A",
    "FXI3SAB_VOR_MM_A",
    "FXI3SAB_VOR_MMAX_A",
    "FXIAB_VOR_MM_A",
    "FXIAB_VOR_MMAX_A",
    "FXYAB_VOR_MM_A",
    "FXYAB_VOR_MMAX_A",
    "FFM_VOR_COM_MM_A_Y",
    "FFM_VOR_COM_MMAX_A_Y",
    "FXI3SAB_VOR_COM_MM_A_Y",
    "FXI3SAB_VOR_COM_MMAX_A_Y",
    "NBJRR50_MM_A",
    "NBJRR50_MMAX_A",
    "NBJRR50_MSOM_A",
    "NBJRR1_MM_A",
    "NBJRR1_MMAX_A",
    "NBJRR1_MSOM_A",
    "NBJRR5_MM_A",
    "NBJRR5_MMAX_A",
    "NBJRR5_MSOM_A",
    "NBJRR10_MM_A",
    "NBJRR10_MMAX_A",
    "NBJRR10_MSOM_A",
    "NBJRR30_MM_A",
    "NBJRR30_MMAX_A",
    "NBJRR30_MSOM_A",
    "NBJRR100_MM_A",
    "NBJRR100_MMAX_A",
    "NBJRR100_MSOM_A",
    "RR_VOR_MM_A",
    "RR_VOR_MMAX_A",
    "RRAB_VOR_MM_A",
    "RRAB_VOR_MMAX_A",
]
# ordinals += ["AN_EXERC"]
ordinals += ["TAILLE1", "TAILLE2"]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface",
            "Surface de moins d",
            "Surface entre 501",
            "Surface entre 1001",
            "Surface entre 1501",
            "Surface de plus de",
        ],
        "SURFACE4": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "SURFACE6": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu",
            "<0.05",
            "0.05-0.1",
            "0.1-0.2",
            "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# get the columns that are to be removed
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += [
    "DEROG13",
    "DEROG14",
    "DEROG16",
]  # no values
# columns that were not available at the time of the frequency model
to_remove += [
    "DEROG13_formatted",
    "DEROG8_formatted",
    "DEROG3_formatted",
    "DEROG16_formatted",
    "DEROG14_formatted",
    "KAPITAL_MAX",
    "KAPITAL_SUM",
]
to_remove += ["IND_Y1_Y2_num", "IND_INC_num"]

# removing columns
categorical_columns = [
    col
    for col in categorical_columns
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col
    for col in numerical_columns
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

195 113 238 546


In [8]:
for c in categorical_columns:
    print(f"'{c}': {np.sort(np.unique([v for v in data[c].value_counts().index]))}")

'ACTIVIT2': ['ACT1' 'ACT2' 'ACT3' 'ACT4' 'ACT5' 'ACT6' 'ACT7' 'ACT8' 'ACT9']
'VOCATION': ['VOC1' 'VOC2' 'VOC3' 'VOC4' 'VOC5' 'VOC6' 'VOC7' 'VOC8']
'ADOSS': ['N' 'O']
'CARACT1': ['N' 'O' 'R']
'CARACT3': ['N' 'O' 'R']
'INDEM1': ['N' 'O']
'TYPBAT1': ['gibier-plumes' 'lapin' 'porcs' 'veaux' 'volaille']
'INDEM2': ['CLASS1' 'CLASS10' 'CLASS11' 'CLASS13' 'CLASS2' 'CLASS3' 'CLASS4'
 'CLASS5' 'CLASS6' 'CLASS7' 'CLASS8' 'CLASS9']
'FRCH1': ['0' '1' '2' '3' 'i']
'FRCH2': ['1' '2' '3' '4' '5' 'A']
'DEROG2': ['N' 'O']
'DEROG3': ['N' 'O']
'DEROG4': ['N' 'O']
'DEROG5': ['N' 'O']
'DEROG8': ['N' 'O']
'DEROG12': ['D03' 'D04' 'D12' 'D18' 'D35']


'KAPITAL34': ['N' 'O']


'KAPITAL35': ['N' 'O']
'KAPITAL37': ['N' 'O']
'KAPITAL40': ['N' 'O']
'KAPITAL41': ['N' 'O']
'KAPITAL42': ['N' 'O']
'KAPITAL43': ['N' 'O']
'RISK6': ['A' 'N' 'O']
'RISK8': ['N' 'O']
'RISK9': ['N' 'O' 'R']
'RISK10': ['N' 'O' 'R']
'RISK11': ['N' 'O' 'R']
'RISK12': ['N' 'O' 'R']
'RISK13': ['N' 'O' 'R']
'EQUIPEMENT2': ['N' 'O' 'R']
'EQUIPEMENT5': ['AP' 'AX' 'CS' 'CX' 'GC' 'GM' 'GU' 'TO']
'ESPINSEE': ['ESP1' 'ESP2' 'ESP3' 'ESP4']
'AN_EXERC': ['ANNEE1' 'ANNEE2' 'ANNEE3' 'ANNEE4' 'ANNEE5' 'ANNEE6' 'ANNEE7' 'ANNEE8'
 'ANNEE9']
'ZONE': ['01' '02' '03' '04' '05' '06' '07' '08' '09' '10' '11' '12' '13' '14'
 '15' '16' '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28'
 '29' '30' '31' '32' '33' '34' '35' '36' '37' '38' '39' '40' '41' '42'
 '43' '44' '45' '46' '47' '48' '49' '50' '51' '52' '53' '54' '55' '56'
 '57' '58' '59' '60' '61' '63' '64' '65' '66' '67' '68' '69' '70' '71'
 '72' '73' '74' '75' '76' '77' '79' '80' '81' '82' '83' '84' '85' '86'
 '87' '88' '89' '90' '91' '92' '93' '94' '9

'LOG_SOC_num_LOG_TOT': [0.0106383  0.01075269 0.01086957 0.01098901 0.01111111 0.01190476
 0.01204819 0.01219512 0.01234568 0.01351351 0.01369863 0.01388889
 0.01408451 0.10638298 0.10752688 0.10869565 0.10989011 0.11904762
 0.12048193 0.12195122 0.12345679 0.125      0.13513514 0.1369863
 0.13888889 0.14084507 0.21505376 0.2173913  0.21978022 0.24096386
 0.24390244 0.24691358 0.2739726  0.27777778 0.28169014 0.32608696
 0.32967033 0.36144578 0.36585366 0.37037037 0.4109589  0.41666667
 0.42253521 0.43956044 0.48192771 0.48780488 0.49382716 0.53763441
 0.54347826 0.54794521 0.55555556 0.56338028 0.60240964 0.6097561
 0.61728395 0.65217391 0.69444444 0.73170732]


'LOG_VETUSTE': [11.27659574 11.42857143 11.62162162 13.22580645 13.61445783 14.10958904
 15.37634409 16.02409639 16.84931507 17.52688172 18.04878049 18.43373494
 19.03614458 19.16666667 19.34782609 19.5890411  19.67741935 20.10869565
 20.48780488 20.84337349 21.34146341 21.52173913 21.82795699 21.94444444
 22.2826087  22.32876712 22.91666667 22.92682927 23.25301205 23.69565217
 23.7804878  23.97849462 24.17582418 24.45652174 24.72222222 25.06849315
 25.36585366 25.6626506  25.69444444 25.86956522 25.92592593 26.2195122
 26.37362637 26.63043478 27.46987952 27.5        27.7173913  27.80487805
 27.80821918 28.04347826 28.07228916 28.16901408 28.27956989 28.39506173
 28.47222222 28.57142857 28.65853659 28.80434783 29.8630137  29.87804878
 29.89130435 30.24390244 30.27777778 30.76923077 30.86419753 30.97826087
 30.98591549 31.09756098 31.25       31.86813187 32.06521739 32.31707317
 32.63888889 32.68292683 32.96703297 33.01369863 33.05555556 33.33333333
 33.53658537 33.8028169  34.02777778 

...

'NBJTNS20_MM_A_NBJTNS20_MMAX_A': ['01. <= 0_01. <= 1' '01. <= 0_02. <= 3' '02. <= 1_02. <= 3'
 '02. <= 1_03. <= 7' '03. <= 2_03. <= 7' '03. <= 2_04. >= 7'
 '04. >= 2_04. >= 7']
'NBJTMS24_MM_A_NBJTMS24_MMAX_A': ['01. <= 1_01. <= 4' '01. <= 1_02. <= 7' '02. <= 1_01. <= 4'
 '02. <= 1_02. <= 7' '02. <= 1_03. <= 14' '03. <= 3_02. <= 7'
 '03. <= 3_03. <= 14' '03. <= 3_04. >= 14' '04. >= 3_04. >= 14']
'TAMPLIAB_VOR_MM_A_TAMPLIAB_VOR_MMAX_A': ['01. <= 7_01. <= 9' '01. <= 7_02. <= 19' '02. <= 15_02. <= 19'
 '02. <= 15_03. <= 22' '03. <= 18_02. <= 19' '03. <= 18_03. <= 22'
 '03. <= 18_04. >= 22' '04. >= 18_03. <= 22' '04. >= 18_04. >= 22']
'TAMPLIM_VOR_MM_A_TAMPLIM_VOR_MMAX_A': ['01. <= 4_01. <= 5' '02. <= 9_02. <= 12' '02. <= 9_03. <= 14'
 '03. <= 10_02. <= 12' '03. <= 10_03. <= 14' '03. <= 10_04. >= 14'
 '04. >= 10_03. <= 14' '04. >= 10_04. >= 14']
'TM_VOR_MM_A_TM_VOR_MMAX_A': ['01. <= 5_01. <= 9' '01. <= 5_02. <= 19' '02. <= 11_01. <= 9'
 '02. <= 11_02. <= 19' '02. <= 11_03. <= 21' '02. <= 11

'TNMAX_VOR_MM_A_TNMAX_VOR_MMAX_A': ['01. <= 5_01. <= 9' '01. <= 5_02. <= 19' '02. <= 11_01. <= 9'
 '02. <= 11_02. <= 19' '02. <= 11_03. <= 20' '03. <= 13_02. <= 19'
 '03. <= 13_03. <= 20' '03. <= 13_04. >= 20' '04. >= 13_02. <= 19'
 '04. >= 13_03. <= 20' '04. >= 13_04. >= 20']
'TX_VOR_MM_A_TX_VOR_MMAX_A': ['01. <= 7_01. <= 12' '01. <= 7_02. <= 24' '02. <= 15_01. <= 12'
 '02. <= 15_02. <= 24' '02. <= 15_03. <= 27' '02. <= 15_04. >= 27'
 '03. <= 18_02. <= 24' '03. <= 18_03. <= 27' '03. <= 18_04. >= 27'
 '04. >= 18_03. <= 27' '04. >= 18_04. >= 27']
'TXAB_VOR_MM_A_TXAB_VOR_MMAX_A': ['01. <= 11_01. <= 16' '01. <= 11_02. <= 32' '02. <= 22_02. <= 32'
 '02. <= 22_03. <= 35' '02. <= 22_04. >= 35' '03. <= 25_02. <= 32'
 '03. <= 25_03. <= 35' '03. <= 25_04. >= 35' '04. >= 25_03. <= 35'
 '04. >= 25_04. >= 35']
'TXMIN_VOR_MM_A_TXMIN_VOR_MMAX_A': ['01. <= 6_01. <= 9' '01. <= 6_02. <= 19' '01. <= 6_03. <= 22'
 '02. <= 9_02. <= 19' '02. <= 9_03. <= 22' '02. <= 9_04. >= 22'
 '03. <= 11_02. <= 19' '03. 

'NBJFF16_MM_A_NBJFF16_MMAX_A': ['01. <= 3_01. <= 8' '01. <= 3_02. <= 10' '02. <= 5_01. <= 8'
 '02. <= 5_02. <= 10' '02. <= 5_03. <= 14' '03. <= 7_02. <= 10'
 '03. <= 7_03. <= 14' '03. <= 7_04. >= 14' '04. >= 7_03. <= 14'
 '04. >= 7_04. >= 14']
'NBJFF28_MM_A_NBJFF28_MMAX_A': ['01. <= 0_01. <= 1' '01. <= 0_02. <= 2' '01. <= 0_03. <= 2'
 '02. <= 1_01. <= 1' '02. <= 1_02. <= 2' '02. <= 1_03. <= 2'
 '03. <= 1_01. <= 1' '03. <= 1_02. <= 2' '03. <= 1_03. <= 2'
 '03. <= 1_04. >= 2' '04. >= 1_02. <= 2' '04. >= 1_03. <= 2'
 '04. >= 1_04. >= 2']
'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A': ['01. <= 11_01. <= 18' '01. <= 11_02. <= 20' '02. <= 13_01. <= 18'
 '02. <= 13_02. <= 20' '02. <= 13_03. <= 22' '03. <= 16_02. <= 20'
 '03. <= 16_03. <= 22' '03. <= 16_04. >= 22' '04. >= 16_03. <= 22'
 '04. >= 16_04. >= 22']
'NBJFXI3S16_MM_A_NBJFXI3S16_MMAX_A': ['01. <= 3_01. <= 6' '01. <= 3_02. <= 9' '02. <= 4_01. <= 6'
 '02. <= 4_02. <= 9' '02. <= 4_03. <= 12' '03. <= 6_02. <= 9'
 '03. <= 6_03. <= 12' '03. <= 6_04. >

'FFM_VOR_COM_MM_A_Y_FFM_VOR_COM_MMAX_A_Y': ['01. <= 1_01. <= 2' '01. <= 1_02. <= 4' '02. <= 3_01. <= 2'
 '02. <= 3_02. <= 4' '02. <= 3_03. <= 5' '03. <= 4_02. <= 4'
 '03. <= 4_03. <= 5' '03. <= 4_04. >= 5' '04. >= 4_03. <= 5'
 '04. >= 4_04. >= 5']
'FXI3SAB_VOR_COM_MM_A_Y_FXI3SAB_VOR_COM_MMAX_A_Y': ['01. <= 4_01. <= 5' '01. <= 4_02. <= 17' '02. <= 12_01. <= 5'
 '02. <= 12_02. <= 17' '02. <= 12_03. <= 30' '03. <= 23_02. <= 17'
 '03. <= 23_03. <= 30' '04. >= 23_03. <= 30' '04. >= 23_04. >= 30']
'NBJRR50_MM_A_NBJRR50_MMAX_A': ['01. <= 0_01. <= 1' '01. <= 0_02. <= 1' '01. <= 0_03. <= 1'
 '02. <= 0_02. <= 1' '02. <= 0_03. <= 1' '03. <= 0_02. <= 1'
 '03. <= 0_03. <= 1' '03. <= 0_04. >= 1' '04. >= 0_03. <= 1'
 '04. >= 0_04. >= 1']
'NBJRR1_MM_A_NBJRR1_MMAX_A': ['01. <= 8_01. <= 13' '01. <= 8_02. <= 17' '02. <= 10_01. <= 13'
 '02. <= 10_02. <= 17' '02. <= 10_03. <= 19' '02. <= 10_04. >= 19'
 '03. <= 11_02. <= 17' '03. <= 11_03. <= 19' '03. <= 11_04. >= 19'
 '04. >= 11_03. <= 19' '04. >= 11_04. >

'VENT_x_CASERNES': ['1.0__01. <= 1' '1.0__02. <= 3' '1.0__03. <= 14' '1.0__04. >= 14'
 '2.0__01. <= 1' '2.0__02. <= 3' '2.0__03. <= 14' '2.0__04. >= 14'
 '3.0__01. <= 1' '3.0__02. <= 3' '3.0__03. <= 14' '3.0__04. >= 14']
'TYPERS': [1 2]


## Processing Qualitative Features

For a binary target variable, following processing is applied:
* Pre-processing of ordinals:
    - ordering modalities according to user-provided values
    - grouping modalities with less than `min_freq=2%` frequency into their closest modality (previous or next modality) according to target rate (train sample)
* Pre-processing of categoricals:
    - grouping modalities with less than `min_freq=2%` frequency into a dedicated one (train sample)
    - ordering modalities according to target rate (train sample)
* All combinations of up to `max_n_mod=5` modalities are sorted by Tschuprow's T with the target variable (train sample)
* Robustness of each combination is put to test (dev sample) 
    - representativness of modalities (more than `min_freq/2=1%` frequency)
    - distinct target rate per consecutive modalities
    - no inversion of target rates between train and dev modalities

For multiclass target variables, the binary-oriented processing steps are applied to each class of the target variable with a One vs Rest approach (except for one of the classes)

In [9]:
import time

_t0 = time.perf_counter()

In [10]:
from AutoCarver import Features, MulticlassCarver

# defining the features to carve
features = Features(categoricals=categorical_columns, ordinals=ordinal_columns)

# defining the carver
carver = MulticlassCarver(
    features=features,
    min_freq=0.02,
    max_n_mod=5,
    dropna=False,
    copy=False,
    verbose=False,
)

# carving train data and testing robustness on dev data
x_train = carver.fit_transform(
    x_train, y_transform(y_train), X_dev=x_dev, y_dev=y_transform(y_dev)
)

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

...

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

In [11]:
print(
    "[2025] qualitative carving (single-process): %.1fs" % (time.perf_counter() - _t0)
)

[2025] qualitative carving (single-process): 2842.3s


In [12]:
# version = "010"
# carver.save(f"model/frequency/{version}_carver.json", light_mode=True)

In [13]:
from AutoCarver import MulticlassCarver

version = "010"
carver = MulticlassCarver.load(f"model/frequency/{version}_carver.json")
carver.summary

content  \
feature                          target_mean label                                                      
Categorical('ACTIVIT2__y=1')     0.000000    0                                     [ACT3, ACT8, ACT2]   
                                 0.000220    1                                           [ACT9, ACT1]   
                                 0.000438    2                    [ACT4, ACT6, ACT7, __OTHER__, ACT5]   
Categorical('VOCATION__y=1')     0.000010    0        [VOC7, VOC5, VOC3, VOC2, __OTHER__, VOC8, VOC1]   
                                 0.000346    1                                           [VOC6, VOC4]   
...                                                                                               ...   
Ordinal('LOG_SOC__y=2')          0.000840    1      [04. <= 40, 05. <= 50, 06. <= 60, 07. <= 70, 0...   
Ordinal('DISTANCE_142__y=2')     0.000303    0                                  [02. <= 17, 01. <= 8]   
                                 0.000575    1                               [04. >= 946, 03. <= 946]   
Ordinal('FXI3SAB_VOR_MM_A__y=2') 0.000364    0                                               01. <= 8   
                                 0.000247    1                      [04. >= 24, 03. <= 24, 02. <= 17]   

                                                    frequency  
feature                          target_mean label             
Categorical('ACTIVIT2__y=1')     0.000000    0       0.066037  
                                 0.000220    1       0.785023  
                                 0.000438    2       0.148940  
Categorical('VOCATION__y=1')     0.000010    0       0.322512  
                                 0.000346    1       0.677488  
...                                                       ...  
Ordinal('LOG_SOC__y=2')          0.000840    1       0.023269  
Ordinal('DISTANCE_142__y=2')     0.000303    0       0.375824  
                                 0.000575    1       0.056698  
Ordinal('FXI3SAB_VOR_MM_A__y=2') 0.000364    0       0.340072  
                                 0.000247    1       0.092451  

[1342 rows x 2 columns]

## Processing Quantitative Features

For a binary target variable, following processing is applied:
* Pre-processing of quantitatives:
    - cutting feature into quantiles of sizes of at least `min_freq/2=5%` (train sample)
    - ordering modalities according to natural order
* All combinations of up to `max_n_mod=5` modalities are sorted by Tschuprow's T with the target variable (train sample)
* Robustness of each combination is put to test (dev sample) 
    - representativness of modalities (more than `min_freq/2=5%` frequency)
    - distinct target rate per consecutive modalities
    - no inversion of target rates between train and dev modalities

For multiclass target variables, the binary-oriented processing steps are applied to each class of the target variable with a One vs Rest approach (except for one of the classes)

In [14]:
import time

_t0 = time.perf_counter()

In [15]:
from AutoCarver import Features, MulticlassCarver

features = Features(quantitatives=numerical_columns)

carver_numericals = MulticlassCarver(
    features=features,
    min_freq=0.10,
    max_n_mod=5,
    dropna=False,
    copy=False,
    verbose=False,
)
x_train = carver_numericals.fit_transform(
    x_train, y_transform(y_train), X_dev=x_dev, y_dev=y_transform(y_dev)
)

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X = X.assign(**casted_columns)
<repo>/.venv-705\Lib\site-packages\AutoCarver\discretizers\utils\base_discretizer.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

In [16]:
print(
    "[2025] quantitative carving (single-process): %.1fs" % (time.perf_counter() - _t0)
)

[2025] quantitative carving (single-process): 383.4s
